# Reinforced Iron Plates

**Type:** Consumer — pulls from storage, stores output in dimensional depot  
**Blueprint:** `reinforced-iron-plate` (90 Iron Plates/min + 180 Screws/min → 15 Reinforced Iron Plates/min at 100%)  
**Pulls from:** Iron Plates storage, Screws storage

> Set supply rates below to `[producer output] − [stash rate]`:
> - Iron Plates: check `iron-plates.ipynb` → use `available_for_reinforced`
> - Screws: check `screws.ipynb` → use the remaining portion of `available_for_downstream`  
>   (after Rotors has taken its share)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from blueprints import BLUEPRINTS, Blueprint, Stage
from satisfactory import ITEMS
import pandas as pd

BP  = BLUEPRINTS
TOL = 0.5


def machine_hover(bp: Blueprint, output_key: str, target_rate: float) -> str:
    if not bp.stages:
        return ""
    rates = bp.stage_rates(output_key, target_rate)
    lines = ["── Machines (per copy) ──"]
    for r in rates:
        name = ITEMS[r["item_key"]].name
        lines.append(f"  {r['machines']}× {r['building'].title()} → {r['per_machine_rate']:.2f} {name}/min each")
    return "<br>" + "<br>".join(lines)

## Constants

In [ ]:
# ── Supply from storage ───────────────────────────────────────────────────────
# These are the rates this chain PULLS from the dimensional depot each minute.
# Must be less than what the producer notebooks output (leaving the stash rate accumulating).
IRON_PLATES_FROM_STORAGE = None  # ← Iron Plates/min pulled from storage
SCREWS_FROM_STORAGE      = None  # ← Screws/min pulled from storage

# ── Blueprint placement ───────────────────────────────────────────────────────
REINFORCED_PLATE_COPIES      = None  # ← number of blueprints to place
REINFORCED_PLATE_OUTPUT_EACH = None  # ← output rate per copy to set in-game (Reinforced Iron Plates/min)

## Derived rates and balance

In [ ]:
assert None not in (IRON_PLATES_FROM_STORAGE, SCREWS_FROM_STORAGE,
                    REINFORCED_PLATE_COPIES, REINFORCED_PLATE_OUTPUT_EACH), \
    "Fill in all constants in the cell above before running this cell."

reinforced_total  = REINFORCED_PLATE_COPIES * REINFORCED_PLATE_OUTPUT_EACH
plates_consumed   = reinforced_total * BP['reinforced-iron-plate'].input_ratio('iron-plate', 'reinforced-iron-plate')
screws_consumed   = reinforced_total * BP['reinforced-iron-plate'].input_ratio('screw',      'reinforced-iron-plate')

assert abs(plates_consumed - IRON_PLATES_FROM_STORAGE) < TOL, (
    f"Iron Plate balance: consuming {plates_consumed:.2f}/min, available {IRON_PLATES_FROM_STORAGE:.2f}/min "
    f"(delta {plates_consumed - IRON_PLATES_FROM_STORAGE:+.2f})"
)
assert abs(screws_consumed - SCREWS_FROM_STORAGE) < TOL, (
    f"Screw balance: consuming {screws_consumed:.2f}/min, available {SCREWS_FROM_STORAGE:.2f}/min "
    f"(delta {screws_consumed - SCREWS_FROM_STORAGE:+.2f})"
)

print(f"✓ Chain balanced")
print(f"  {IRON_PLATES_FROM_STORAGE:.2f} Iron Plates/min  +  {SCREWS_FROM_STORAGE:.2f} Screws/min")
print(f"  →  {reinforced_total:.2f} Reinforced Iron Plates/min")
print(f"  Set each of {REINFORCED_PLATE_COPIES} blueprint{'s' if REINFORCED_PLATE_COPIES > 1 else ''} to: {REINFORCED_PLATE_OUTPUT_EACH:.2f} Reinforced Iron Plates/min")